# 🎧 Projeto Caio.Terca.Aline — Spotify Tracks
## Inteligência Artificial e Machine Learning

**Objetivo:** prever o gênero musical (`track_genre`) usando características de áudio das faixas. O notebook segue os conteúdos estudados: exploração de dados, preparação, treino/validação/teste, baseline, classificação, pipeline, comparação de modelos e avaliação.

**Base:** [Spotify Tracks Dataset — Kaggle](https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset)

A base possui aproximadamente 114 mil registros. Uma mesma faixa pode aparecer em mais de um gênero. Por isso, a separação será feita por `track_id`, evitando que a mesma faixa apareça em conjuntos diferentes.

## 1. Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from google.colab import files

from sklearn.model_selection import GroupShuffleSplit
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, ConfusionMatrixDisplay
)

pd.set_option("display.max_columns", None)

## 2. Carregamento da base
Baixe o CSV na página do Kaggle e selecione o arquivo quando a janela abrir.

In [ ]:
uploaded = files.upload()
arquivo = next(iter(uploaded))
df = pd.read_csv(arquivo)
print("Arquivo:", arquivo)

## 3. Conhecendo os dados

In [ ]:
display(df.head())
print("Dimensões:", df.shape)
print("Colunas:", df.columns.tolist())
print("\nTipos:")
print(df.dtypes)
print("\nInformações:")
df.info()

In [ ]:
print("Valores ausentes:")
display(df.isnull().sum().sort_values(ascending=False))
print("Duplicatas completas:", df.duplicated().sum())
print("Faixas diferentes:", df["track_id"].nunique())
print("Gêneros:", df["track_genre"].nunique())

## 4. Preparação inicial
Removemos apenas linhas totalmente duplicadas. `track_id` será usado para separar os dados, mas não será entrada do modelo.

In [ ]:
df = df.drop_duplicates().copy()

if df["explicit"].dtype == "object":
    df["explicit"] = (
        df["explicit"].astype(str).str.lower()
        .map({"true": 1, "false": 0, "1": 1, "0": 0})
    )
else:
    df["explicit"] = df["explicit"].astype(int)

df["duration_min"] = df["duration_ms"] / 60000
print("Dimensões após limpeza:", df.shape)

## 5. Análise exploratória
### 5.1 Estatísticas das variáveis numéricas

In [ ]:
colunas_numericas = df.select_dtypes(include=np.number).columns
display(df[colunas_numericas].describe().T)

### 5.2 Quantidade de registros por gênero

In [ ]:
contagem_generos = df["track_genre"].value_counts()
plt.figure(figsize=(14, 5))
contagem_generos.head(20).plot(kind="bar")
plt.title("20 gêneros com mais registros")
plt.xlabel("Gênero")
plt.ylabel("Quantidade")
plt.xticks(rotation=65, ha="right")
plt.tight_layout()
plt.show()

### 5.3 Distribuição da popularidade

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df["popularity"].dropna(), bins=20, edgecolor="black")
plt.title("Distribuição da popularidade")
plt.xlabel("Popularidade (0 a 100)")
plt.ylabel("Quantidade de faixas")
plt.tight_layout()
plt.show()

display(
    df.groupby("track_genre")["popularity"]
      .mean().sort_values(ascending=False)
      .head(15).to_frame("popularidade_media")
)

### 5.4 Características musicais

In [ ]:
caracteristicas = [
    "danceability", "energy", "valence",
    "acousticness", "speechiness", "instrumentalness"
]
df[caracteristicas].hist(figsize=(12, 8), bins=25, edgecolor="black")
plt.suptitle("Distribuição das características musicais")
plt.tight_layout()
plt.show()

In [ ]:
correlacao = df[colunas_numericas].corr(numeric_only=True)
plt.figure(figsize=(11, 8))
plt.imshow(correlacao, aspect="auto")
plt.colorbar(label="Correlação")
plt.xticks(range(len(correlacao.columns)), correlacao.columns, rotation=90)
plt.yticks(range(len(correlacao.index)), correlacao.index)
plt.title("Correlação entre variáveis numéricas")
plt.tight_layout()
plt.show()

## 6. Definição do problema

- **Aprendizado:** supervisionado.
- **Tarefa:** classificação multiclasse.
- **Alvo (`y`):** `track_genre`.
- **Entradas (`X`):** características de áudio e indicador `explicit`.
- **Excluídos:** nomes, artista, álbum, identificador e popularidade.

A popularidade não será usada como entrada, pois o objetivo é classificar o gênero a partir das características musicais.

In [ ]:
colunas_entrada = [
    "duration_ms", "explicit", "danceability", "energy", "key",
    "loudness", "mode", "speechiness", "acousticness",
    "instrumentalness", "liveness", "valence", "tempo",
    "time_signature"
]
X = df[colunas_entrada].copy()
y = df["track_genre"].copy()
grupos = df["track_id"].copy()

print("Entradas:", X.shape)
print("Número de gêneros:", y.nunique())

## 7. Separação em treino, validação e teste

- **Treino:** aprende os padrões.
- **Validação:** compara os modelos e ajuda a escolher.
- **Teste:** fica reservado para a avaliação final.

A divisão por `track_id` impede que registros da mesma faixa apareçam em conjuntos diferentes. `random_state=42` torna a divisão reproduzível.

In [ ]:
# Separa 15% para teste final
split_teste = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
idx_dev, idx_teste = next(split_teste.split(X, y, groups=grupos))

X_dev, y_dev = X.iloc[idx_dev], y.iloc[idx_dev]
g_dev = grupos.iloc[idx_dev]
X_teste, y_teste = X.iloc[idx_teste], y.iloc[idx_teste]
g_teste = grupos.iloc[idx_teste]

# Separa aproximadamente 15% do total para validação
split_validacao = GroupShuffleSplit(n_splits=1, test_size=0.1765, random_state=42)
idx_treino, idx_valid = next(
    split_validacao.split(X_dev, y_dev, groups=g_dev)
)

X_treino, y_treino = X_dev.iloc[idx_treino], y_dev.iloc[idx_treino]
X_validacao, y_validacao = X_dev.iloc[idx_valid], y_dev.iloc[idx_valid]

print("Treino:", X_treino.shape)
print("Validação:", X_validacao.shape)
print("Teste:", X_teste.shape)

g_treino = g_dev.iloc[idx_treino]
g_validacao = g_dev.iloc[idx_valid]
assert set(g_treino).isdisjoint(set(g_validacao))
assert set(g_treino).isdisjoint(set(g_teste))
assert set(g_validacao).isdisjoint(set(g_teste))
print("Sem track_id compartilhado entre conjuntos.")

### 7.1 Distribuição dos gêneros por conjunto

In [ ]:
distribuicao = pd.DataFrame({
    "treino": y_treino.value_counts(normalize=True),
    "validacao": y_validacao.value_counts(normalize=True),
    "teste": y_teste.value_counts(normalize=True)
}).fillna(0)
display(distribuicao)

## 8. Pipeline de preparação
O `SimpleImputer` trata valores ausentes; o `StandardScaler` padroniza as variáveis. O pipeline mantém a preparação e o modelo no mesmo fluxo.

In [ ]:
def criar_pipeline(classificador):
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("modelo", classificador)
    ])

## 9. Baseline
A baseline prevê sempre o gênero mais frequente no treino. Os modelos precisam superar essa referência.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_treino, y_treino)
pred_baseline = baseline.predict(X_validacao)
print("Acurácia da baseline:", accuracy_score(y_validacao, pred_baseline))

## 10. Treinamento e comparação

Serão comparados:
1. **Regressão logística:** combina as características por pesos.
2. **Árvore de decisão:** aprende regras de divisão.
3. **MLP:** rede neural multicamadas com ativação ReLU.

A métrica principal será o **F1-macro**, que calcula o F1 de cada gênero e depois faz a média, atribuindo o mesmo peso às classes.

In [ ]:
modelos = {
    "Regressão logística": criar_pipeline(
        LogisticRegression(max_iter=400, solver="lbfgs")
    ),
    "Árvore de decisão": criar_pipeline(
        DecisionTreeClassifier(
            max_depth=20, min_samples_leaf=3, random_state=42
        )
    ),
    "Rede neural MLP": criar_pipeline(
        MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation="relu",
            max_iter=25,
            early_stopping=True,
            n_iter_no_change=5,
            random_state=42,
            batch_size=1024
        )
    )
}

resultados = []
modelos_treinados = {}

for nome, modelo in modelos.items():
    print("Treinando:", nome)
    modelo.fit(X_treino, y_treino)
    previsoes = modelo.predict(X_validacao)

    acc = accuracy_score(y_validacao, previsoes)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_validacao, previsoes, average="macro", zero_division=0
    )
    resultados.append({
        "Modelo": nome,
        "Acurácia": acc,
        "Precisão macro": prec,
        "Recall macro": rec,
        "F1-macro": f1
    })
    modelos_treinados[nome] = modelo

comparacao = pd.DataFrame(resultados).sort_values(
    "F1-macro", ascending=False
)
display(comparacao)

## 11. Escolha pela validação
O modelo com maior F1-macro na validação será escolhido. O teste ainda não foi usado.

In [ ]:
melhor_nome = comparacao.iloc[0]["Modelo"]
melhor_modelo = modelos_treinados[melhor_nome]
print("Modelo escolhido:", melhor_nome)

## 12. Avaliação final no teste

In [ ]:
pred_teste = melhor_modelo.predict(X_teste)

acc = accuracy_score(y_teste, pred_teste)
prec, rec, f1, _ = precision_recall_fscore_support(
    y_teste, pred_teste, average="macro", zero_division=0
)

print("Modelo:", melhor_nome)
print(f"Acurácia: {acc:.4f}")
print(f"Precisão macro: {prec:.4f}")
print(f"Recall macro: {rec:.4f}")
print(f"F1-macro: {f1:.4f}")

In [ ]:
print(classification_report(y_teste, pred_teste, zero_division=0))

## 13. Matriz de confusão
Como há muitos gêneros, exibiremos os 15 mais frequentes no teste para facilitar a leitura. As métricas gerais consideram todas as classes.

In [ ]:
top_classes = y_teste.value_counts().head(15).index.tolist()
mascara = y_teste.isin(top_classes)

fig, ax = plt.subplots(figsize=(12, 10))
ConfusionMatrixDisplay.from_predictions(
    y_teste[mascara],
    pd.Series(pred_teste, index=y_teste.index)[mascara],
    labels=top_classes,
    xticks_rotation=90,
    cmap="Blues",
    ax=ax,
    colorbar=False
)
plt.title("Matriz de confusão — 15 gêneros mais frequentes")
plt.tight_layout()
plt.show()

## 14. Importância das características
A árvore de decisão fornece uma medida de importância das variáveis. Ela indica quais características foram mais utilizadas nas divisões, mas não comprova causalidade.

In [ ]:
arvore = modelos_treinados["Árvore de decisão"].named_steps["modelo"]
importancias = pd.Series(
    arvore.feature_importances_, index=colunas_entrada
).sort_values(ascending=False)

display(importancias.to_frame("importancia"))
plt.figure(figsize=(9, 5))
importancias.sort_values().plot(kind="barh")
plt.title("Importância das características — árvore de decisão")
plt.xlabel("Importância")
plt.tight_layout()
plt.show()

## 15. Conclusão

Após executar o notebook, responda:

- Qual modelo teve o melhor F1-macro na validação?
- Ele superou a baseline?
- Quais foram a acurácia e o F1-macro no teste?
- Quais gêneros foram mais confundidos?
- Quais características tiveram maior importância na árvore?
- Quais limitações foram observadas?

**Limitações:** o gênero registrado é uma categoria associada à faixa e pode não ser uma classificação única e universal. A mesma faixa pode aparecer em mais de um gênero. Os resultados dependem das faixas e dos gêneros presentes nesta base e não devem ser generalizados automaticamente para todo o catálogo do Spotify.

## Referência
Maharshi Pandya. *Spotify Tracks Dataset*. Kaggle.  
https://www.kaggle.com/datasets/maharshipandya/-spotify-tracks-dataset